## Вектор Пойнтинга и уравнения акустики 1-го порядка

Решение системы уравнений 1-го порядка позволяет напрямую использовать вектор скоростей частиц $\mathbf{v}$ для анализа динамики поля:

$$
\begin{cases}
\frac{\partial p}{\partial t} + \kappa \nabla \cdot \mathbf{v} = f \\
\rho \frac{\partial \mathbf{v}}{\partial t} + \nabla p = 0
\end{cases}
$$

Вектор Пойнтинга ($\mathbf{S}$) определяет плотность и направление потока энергии акустической волны. В изотропной среде он указывает направление движения фронта в каждой точке:

$$\mathbf{S} = p \cdot \mathbf{v} = (p v_x, p v_z)$$

### Практическое применение

Использование $\mathbf{S}$ позволяет разделять волновые поля без трудоёмких интегральных преобразований:

- **RTM (Reverse Time Migration):** сепарация поля на падающее ($S_z > 0$) и отражённое ($S_z < 0$) для подавления артефактов обратного рассеяния.
- **Angle-Domain Imaging:** определение локальных углов падения и отражения для построения угловых сейсмограмм (ADCIGs).
- **ВСП и декомпозиция:** выделение чистой прямой волны или целевых отражений/дифракций по направлению переноса энергии.
- **Анализ освещённости:** оценка плотности энергии, достигающей целевых горизонтов в сложных солевых и подсолевых структурах.

В этом ноутбуке:
1) создадим горизонтально-слоистую модель с дифрактором
2) запустим симуляцию с уравнением 1-го порядка
3) посмотрим анимацию давления с наложенными стрелочками вектора Пойнтинга
4) сделаем разделение волнового поля на падающее и восходящее

In [ ]:
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import matplotlib.pyplot as plt
from tqdm.notebook import tqdm
from bruges.filters import ricker

from ipywidgets import interact, IntSlider
from IPython.display import clear_output, display


### Создание модели

In [ ]:
# Параметры сетки и модель скорости
nx, nz = 301, 201
dx, dz = 5.0, 5.0
tmax_ms = 1000.0

# Простая двухслойная модель с низковеcтностным включением
Vp = np.ones((nz, nx), dtype=np.float32) * 2000.0
Vp[100:, :] = 2400.0
Vp[130:135, 220:225] = 1600.0  # небольшое включение низкой скорости

# Позиция источника и приёмников (в метрах)
sx, sz = nx * dx / 2.0, dz
rx = np.arange(0, nx*dx + dx, 25.0)
rz = 0.0

# Визуализация модели
plt.figure(figsize=(12, 6))
plt.imshow(Vp, cmap='jet', extent=[0, nx*dx, nz*dz, 0], aspect='auto')
plt.colorbar(label='Velocity (m/s)')
plt.scatter(sx, sz, c='red', marker='*', s=100, label='Source')
plt.scatter(rx, np.full_like(rx, rz), c='white', marker='^', s=5, label='Receivers')
plt.xlabel('Distance (m)')
plt.ylabel('Depth (m)')
plt.title('Velocity Model (Vp) with Source and Receivers')
plt.legend()
plt.show()

### Функция для создания поглощающего слоя (упрощенный аналог PML)

In [ ]:
def absorb(nz, nx, thickness):
    """Коэффициенты поглощающего слоя.
    Возвращает массив формы (nz, nx), согласованный с полями p, vx, vz.
    """
    FW = thickness
    a = 0.0053
    coeff = np.exp(-(a ** 2) * np.arange(FW, dtype=np.float32) ** 2)
    absorb_coeff = np.ones((nz, nx), dtype=np.float32)
    # left/right (x-direction) - affect columns
    for i in range(FW):
        absorb_coeff[:, i] *= coeff[FW - 1 - i]
        absorb_coeff[:, nx - 1 - i] *= coeff[FW - 1 - i]
    # top/bottom (z-direction) - affect rows
    for j in range(FW)b:
        absorb_coeff[j, :] *= coeff[FW - 1 - j]
        absorb_coeff[nz - 1 - j, :] *= coeff[FW - 1 - j]
    return absorb_coeff

### Функция для обновления полей давления и скоростей для симуляции 1-го порядка

In [ ]:
def update_velocity_stress_staggered(p, vx, vz, Vp, rho, dt, dx, dz):
    """
    Обновление полей давления и скорости (акустика 1-го порядка)
    с использованием схемы 4-го порядка на шахматной сетке.
    """
    # Коэффициенты для шахматной сетки 4-го порядка
    c1 = 9/8
    c2 = -1/24
    
    # 1. ОБНОВЛЕНИЕ СКОРОСТЕЙ (на основе градиента давления)
    # vx смещена относительно p на +1/2 по x
    # vz смещена относительно p на +1/2 по z
    
    # Расчет vx
    # Используем p в точках (j-1, j, j+1, j+2) для аппроксимации в j+1/2
    vx[2:-2, 2:-2] -= (dt / rho[2:-2, 2:-2]) * (
        c1 * (p[2:-2, 3:-1] - p[2:-2, 2:-2]) / dx +
        c2 * (p[2:-2, 4:]   - p[2:-2, 1:-3]) / (3 * dx)
    )
    
    # Расчет vz
    vz[2:-2, 2:-2] -= (dt / rho[2:-2, 2:-2]) * (
        c1 * (p[3:-1, 2:-2] - p[2:-2, 2:-2]) / dz +
        c2 * (p[4:, 2:-2]   - p[1:-3, 2:-2]) / (3 * dz)
    )
    
    # 2. ОБНОВЛЕНИЕ ДАВЛЕНИЯ (на основе дивергенции скорости)
    # p[i, j] -= dt * rho * Vp^2 * (dvx/dx + dvz/dz)
    
    # Предварительно вычисляем коэффициент K = rho * Vp^2
    K = rho * Vp**2
    
    p[2:-2, 2:-2] -= K[2:-2, 2:-2] * dt * (
        # Производная vx по x в узле давления
        (c1 * (vx[2:-2, 2:-2] - vx[2:-2, 1:-3]) / dx +
         c2 * (vx[2:-2, 3:-1] - vx[2:-2, 0:-4]) / (3 * dx)) +
        
        # Производная vz по z в узле давления
        (c1 * (vz[2:-2, 2:-2] - vz[1:-3, 2:-2]) / dz +
         c2 * (vz[3:-1, 2:-2] - vz[0:-4, 2:-2]) / (3 * dz))
    )

In [ ]:
### Параметры времени и импульс источника

In [ ]:
dt = 0.0005
Tmax = 1000.0
nt = int(Tmax / 1000.0 / dt)
source_freq = 25.0
source_wavelet, t_wavelet = ricker(duration=0.064, dt=dt, f=source_freq)
plt.plot(t_wavelet, source_wavelet)

### Акустическая симуляция 1-го порядка

In [ ]:
# Добавление поглощающих слоев
n_absorb = 50
Vp_padded = np.pad(Vp, n_absorb, mode="edge")
nz_pad, nx_pad = Vp_padded.shape  # Исправлено: (nz, nx)

save_every = 20

# Для простоты, константная плотность
rho = np.ones_like(Vp_padded, dtype=np.float32)

# Позиция источника в padded координатах
isrc = int(round(sx / dx)) + n_absorb
jsrc = int(round(sz / dz)) + n_absorb
isrc = max(1, min(nx_pad - 2, isrc))
jsrc = max(1, min(nz_pad - 2, jsrc))

# Поля давления и скоростей
p = np.zeros((nz_pad, nx_pad), dtype=np.float32)  # Форма (nz, nx)
vx = np.zeros((nz_pad, nx_pad), dtype=np.float32)
vz = np.zeros((nz_pad, nx_pad), dtype=np.float32)

# приёмники: индексы в исходной сетке и в padded-координатах
rx_positions = np.asarray(rx)
rx_ix = np.round(rx_positions / dx).astype(int)
rx_ix = np.clip(rx_ix, 0, nx - 1)
rx_ix_pad = rx_ix + n_absorb

rz_ix = int(round(rz / dz))
rz_ix = np.clip(rz_ix, 0, nz - 1)
rz_ix_pad = rz_ix + n_absorb

# сейсмограммы по реальному времени (nt x n_receivers)
seis_p_time = np.zeros((nt, len(rx_ix)), dtype=np.float32)
seis_vx_time = np.zeros_like(seis_p_time)
seis_vz_time = np.zeros_like(seis_p_time)



# История для сохранения (исходный размер без padding)
nx_crop = nx
nz_crop = nz
n_save = (nt + save_every - 1) // save_every
p_history = np.zeros((n_save, nz_crop, nx_crop), dtype=np.float32)  # Форма (nz, nx)
vx_history = np.zeros((n_save, nz_crop, nx_crop), dtype=np.float32) 
vz_history = np.zeros((n_save, nz_crop, nx_crop), dtype=np.float32)

# Коэффициенты поглощения
absorb_coeff = absorb(nz_pad, nx_pad, n_absorb)

# Индексы для вырезания исходной области
i0, i1 = n_absorb, n_absorb + nx_crop  # x-индексы
j0, j1 = n_absorb, n_absorb + nz_crop  # z-индексы

save_idx = 0
for it in tqdm(range(nt)):
    # Обновление полей
    update_velocity_stress_staggered(p, vx, vz, Vp_padded, rho, dt, dx, dz)
    
    # Источник (взрыв) - добавляем вейвлет
    if it < len(source_wavelet):
        p[jsrc, isrc] += source_wavelet[it] 
    
    # Применение поглощающих границ
    p *= absorb_coeff
    vx *= absorb_coeff
    vz *= absorb_coeff

    # Запись в сейсмограммы
    seis_p_time[it, :]  = p[rz_ix_pad, rx_ix_pad]
    seis_vx_time[it, :] = vx[rz_ix_pad, rx_ix_pad]
    seis_vz_time[it, :] = vz[rz_ix_pad, rx_ix_pad]
    
    # Сохранение снимков
    if it % save_every == 0 and save_idx < n_save:
        # Вырезаем центральную область без padding
        p_history[save_idx] = p[j0:j1, i0:i1]  # Форма (nz, nx)
        vx_history[save_idx] = vx[j0:j1, i0:i1]
        vz_history[save_idx] = vz[j0:j1, i0:i1]
        save_idx += 1

print(f"Симуляция завершена. Сохранено {save_idx} снимков.")

Cейсмограммы всех компонент поля (давление, Z и X проекции скоростей частиц)

In [ ]:
times_ms = np.arange(nt) * dt * 1000.0

fig, axes = plt.subplots(1, 3, figsize=(15, 5))
vmin, vmax = np.percentile(seis_p_time, [2, 98])
axes[0].imshow(seis_p_time, aspect='auto', cmap='gray',
           extent=[0, (len(rx_ix)-1)*25.0, times_ms[-1], times_ms[0]], vmin=vmin, vmax=vmax)
axes[0].set_xlabel('Distance (m)')
axes[0].set_ylabel('Time (ms)')
axes[0].set_title('P Seismogram')
vmin, vmax = np.percentile(seis_vz_time, [2, 98])
axes[1].imshow(seis_vz_time, aspect='auto', cmap='gray',
           extent=[0, (len(rx_ix)-1)*25.0, times_ms[-1], times_ms[0]], vmin=vmin, vmax=vmax)
axes[1].set_xlabel('Distance (m)')
axes[1].set_ylabel('Time (ms)')
axes[1].set_title('Vz Seismogram')
vmin, vmax = np.percentile(seis_vx_time, [2, 98])
axes[2].imshow(seis_vx_time, aspect='auto', cmap='gray',
           extent=[0, (len(rx_ix)-1)*25.0, times_ms[-1], times_ms[0]], vmin=vmin, vmax=vmax)
axes[2].set_xlabel('Distance (m)')
axes[2].set_ylabel('Time (ms)')
axes[2].set_title('Vx Seismogram')    


### Функция для получения вектора Пойнтинга по X и Z с заданным прореживанием 

In [ ]:
def poynting_vector(p, vx, vz, step=10):
    """Вычисление вектора Пойнтинга для акустической волны.

    Возвращает компоненты Sx, Sz, прореженные по обеим осям с шагом `step`.
    """
    Sx = p * vx  # Горизонтальная компонента
    Sz = p * vz  # Вертикальная компонента
    # Прореживание по обоим измерениям: [rows, cols]
    return Sx[::step, ::step], Sz[::step, ::step]

### Визуализация вектора пойнтинга на кадрах давления

Для каждого кадра рассчитаем прореженные значения вектора Пойнтинга и отнормализуем их длину, так как нам важно только направление стрелок. Прореживание и нормализация нужны только для визуализации.
Создание анимации может занять некоторое время (около минуты)

In [ ]:
from matplotlib.animation import FuncAnimation
from IPython.display import HTML
import matplotlib as mpl

# Увеличиваем лимит до 100 МБ (по умолчанию он ~20 МБ)
mpl.rcParams['animation.embed_limit'] = 100.0

# 1. Сетка и лимиты
skip = 10 # шаг прореживания для вектора Пойнтинга
X, Z = np.meshgrid(np.arange(0, nx*dx, skip*dx), np.arange(0, nz*dz, skip*dz))
vmin_p, vmax_p = np.percentile(p_history, 3), np.percentile(p_history, 97)
times_saved_ms = np.arange(len(p_history)) * save_every * dt * 1000.0


# 2. Создание фигуры
fig, ax = plt.subplots(figsize=(9, 6))

im_p = ax.imshow(p_history[0], cmap='gray', aspect='auto', 
                 extent=[0, nx*dx, nz*dz, 0], vmin=vmin_p, vmax=vmax_p)

ax.imshow(Vp, cmap='seismic', aspect='auto', extent=[0, nx*dx, nz*dz, 0],
          alpha=0.3, vmin=Vp.min(), vmax=Vp.max())

q = ax.quiver(X, Z, np.zeros_like(X), np.zeros_like(Z), color='red', 
              pivot='mid', scale=25, width=0.003)

ax.set_xlabel('x (m)')
ax.set_ylabel('z (m)')

# Увеличиваем отступ заголовка (y > 1.0), чтобы освободить место для таймера
ax.set_title('Seismic Wave Propagation & Poynting Vector', y=1.08, fontsize=14)

# Размещаем таймер чуть ниже заголовка, но выше осей
time_text = ax.text(0.5, 1.03, '', transform=ax.transAxes, ha='center', 
                    fontsize=11, color='blue', fontweight='bold')

# 3. Функция обновления
def update(frame_idx):
    # Обновление изображения
    p = p_history[frame_idx]
    im_p.set_data(p)
    
    # Вектор Пойнтинга
    vx = vx_history[frame_idx]
    vz = vz_history[frame_idx]
    sx, sz = poynting_vector(p, vx, vz, step=skip)
    
    mag = np.sqrt(sx**2 + sz**2)
    max_mag = np.max(mag)
    
    if max_mag > 0:
        cutoff = 0.0025 * max_mag 
        mask = mag > cutoff
        sx_norm, sz_norm = np.zeros_like(sx), np.zeros_like(sz)
        sx_norm[mask], sz_norm[mask] = sx[mask]/mag[mask], sz[mask]/mag[mask]
        q.set_UVC(sx_norm, -sz_norm)
    
    # Обновление времени
    t_ms = times_saved_ms[frame_idx]
    time_text.set_text(f'Time: {t_ms:.2f} ms')
    
    return im_p, q, time_text

# 4. Создание анимации на полный объем данных
# cache_frame_data=False экономит память при больших массивах
ani = FuncAnimation(fig, update, frames=range(len(p_history)), 
                    interval=80, blit=True, cache_frame_data=False)

plt.close(fig)

# 5. Вывод
HTML(ani.to_jshtml())

### Простейшее применение вектора Пойнтинга - разделение волны на падающую и восходящую. Полезно для RTM, моделирования ВСП.

Разделение полей давления на падающее и восходящее производится по знаку Sz (вертикальной составляющей вектора пойнтинга). Для падающей волны (Sz > 0) поле с Sz < 0 обнуляется (в простом случае, по хорошему нужно делать какое-то плавное затухание), и наоборот

In [ ]:
vmin_p, vmax_p = np.percentile(p_history, 2), np.percentile(p_history, 98)
ext = [0, nx*dx, nz*dz, 0]

# 2. Создание фигуры с тремя подобластями
fig, (ax_all, ax_down, ax_up) = plt.subplots(1, 3, figsize=(14, 5), sharey=True)

# Инициализация imshow
im_all  = ax_all.imshow(p_history[0],  cmap='gray', aspect='auto', extent=ext, vmin=vmin_p, vmax=vmax_p)
im_down = ax_down.imshow(np.zeros_like(p_history[0]), cmap='gray', aspect='auto', extent=ext, vmin=vmin_p, vmax=vmax_p)
im_up   = ax_up.imshow(np.zeros_like(p_history[0]),   cmap='gray', aspect='auto', extent=ext, vmin=vmin_p, vmax=vmax_p)

# Добавляем подложку Vp и настраиваем оси
for ax in [ax_all, ax_down, ax_up]:
    ax.imshow(Vp, cmap='seismic', aspect='auto', extent=ext, alpha=0.15, vmin=Vp.min(), vmax=Vp.max())
    ax.set_xlabel('x (m)')

ax_all.set_ylabel('z (m)')
ax_all.set_title('Total Pressure Field (P)', pad=20)
ax_down.set_title('Down-going Waves (Sz > 0)', pad=20)
ax_up.set_title('Up-going Waves (Sz < 0)', pad=20)

# Общий таймер сверху по центру
time_text = fig.text(0.5, 0.95, '', ha='center', va='top', fontsize=14, fontweight='bold', color='darkred')

# 3. Функция анимации
def update(frame_idx):
    p = p_history[frame_idx]
    vz = vz_history[frame_idx]
    
    # Расчет Sz для разделения полей
    sz = p * vz
    
    # Маскирование: оставляем только нужные направления
    p_down = np.where(sz > 0, p, 0)
    p_up   = np.where(sz < 0, p, 0)
    
    # Обновление данных
    im_all.set_data(p)
    im_down.set_data(p_down)
    im_up.set_data(p_up)
    
    # Обновление времени
    t_ms = frame_idx * save_every * dt * 1000
    time_text.set_text(f'Time: {t_ms:.1f} ms')
    
    return im_all, im_down, im_up, time_text

# 4. Сборка анимации
# frames=range(0, len(p_history), 2) - можно добавить шаг, если файл слишком тяжелый
ani = FuncAnimation(fig, update, frames=len(p_history), 
                    interval=60, blit=True, cache_frame_data=False)

plt.tight_layout(rect=[0, 0, 1, 0.9])
plt.close(fig)

# 5. Вывод в блокнот
HTML(ani.to_jshtml())